In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin, clone

# Flexible, all-in-one preprocessing function (no train/test split)
def AutoSweep(
    file_path,
    target_column=None,
    drop_columns=None,
    drop_threshold=1.0,
    impute_strategy_num='mean',
    impute_strategy_cat='most_frequent',
    scaler=None,
    encode_categorical=None,
    remove_low_variance=False,
    variance_thresh=0.0,
    # New options: outliers, datetime, target encoding, correlation removal
    outlier_method=None,                # None | 'iqr' | 'zscore'
    outlier_threshold=1.5,              # multiplier for IQR or z-score threshold (z default 3.0 often used)
    cap_outliers=False,                 # if True cap instead of drop
    extract_datetime=False,             # try to parse and extract datetime features
    drop_datetime_original=False,
    target_encode=False,                # target mean encoding for categorical vars (requires target_column)
    remove_correlated=False,
    corr_threshold=0.95,
    # Structured output flag
    structured_output=True,
    verbose=True
):
    """
    Flexible preprocessing for ML tasks. Handles:
    - Loading CSV/Excel
    - Dropping columns (user, constant, high-missing)
    - Imputation (numeric/categorical, multiple strategies)
    - Scaling (Standard, MinMax, Robust)
    - Encoding (OneHot, Ordinal, Label)
    - Low-variance feature removal
    - Optional: outlier detection/capping, datetime extraction, target encoding, correlation-based selection
    - Returns processed DataFrame(s) and feature names or a structured dict containing processing info
    """
    # Initialize info log
    info = {
        'original_shape': None,
        'loaded_shape': None,
        'dropped_missing_columns': [],
        'dropped_user_columns': [],
        'dropped_constant_columns': [],
        'datetime_extracted': [],
        'target_encoded_columns': [],
        'outlier_method': outlier_method,
        'outlier_dropped_count': 0,
        'cap_outliers': cap_outliers,
        'correlated_dropped': [],
        'low_variance_dropped': []
    }
    # Load data
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)
    info['loaded_shape'] = df.shape
    if verbose:
        print(f"Loaded shape: {df.shape}")
    # Drop columns with missing values above threshold
    if drop_threshold < 1.0:
        missing_frac = df.isnull().mean()
        cols_to_drop = missing_frac[missing_frac > drop_threshold].index.tolist()
        df = df.drop(columns=cols_to_drop)
        info['dropped_missing_columns'] = cols_to_drop
        if verbose and cols_to_drop:
            print(f"Dropped columns with >{drop_threshold*100}% missing: {cols_to_drop}")
    else:
        df = df.dropna(axis=1, how='all')
    # Drop duplicate rows
    df = df.drop_duplicates()
    # Drop user-specified columns
    if drop_columns:
        df = df.drop(columns=drop_columns, errors='ignore')
        info['dropped_user_columns'] = [c for c in drop_columns if c in df.columns]
        if verbose:
            print(f"Dropped user columns: {drop_columns}")
    # Remove columns with only one unique value (constant)
    nunique = df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        df = df.drop(columns=constant_cols)
        info['dropped_constant_columns'] = constant_cols
        if verbose:
            print(f"Dropped constant columns: {constant_cols}")
    # Remove rows with all missing values
    df = df.dropna(axis=0, how='all')
    # Strip whitespace from string columns
    for col in df.select_dtypes(include=['object', 'string']):
        df[col] = df[col].astype(str).str.strip()
    # Datetime feature extraction (optional)
    datetime_cols = []
    if extract_datetime:
        for col in df.columns:
            if df[col].dtype.kind in ('M',):
                datetime_cols.append(col)
            elif df[col].dtype == 'object' or df[col].dtype == 'string':
                parsed = pd.to_datetime(df[col], errors='coerce')
                non_na_frac = parsed.notna().mean()
                if non_na_frac > 0.5:  # heuristics: treat as datetime if >50% parseable
                    df[col] = parsed
                    datetime_cols.append(col)
        for col in datetime_cols:
            if verbose:
                print(f"Extracting datetime features from {col}")
            df[f"{col}_year"] = df[col].dt.year
            df[f"{col}_month"] = df[col].dt.month
            df[f"{col}_day"] = df[col].dt.day
            df[f"{col}_weekday"] = df[col].dt.weekday
            df[f"{col}_hour"] = df[col].dt.hour
        if drop_datetime_original and datetime_cols:
            df = df.drop(columns=datetime_cols)
    info['datetime_extracted'] = datetime_cols
    # Separate target if provided (do early so target-encoding & outlier removal can use it)
    y = None
    if target_column and target_column in df.columns:
        y = df[target_column].copy()
        df = df.drop(columns=[target_column])
    # Target encoding (optional) - WARNING: no train/test split here (use carefully)
    if target_encode:
        if y is None:
            raise ValueError("target_encode requires target_column to be provided")
        cat_cols = df.select_dtypes(include=['object','category','string']).columns.tolist()
        for col in cat_cols:
            if verbose:
                print(f"Applying target mean encoding to {col}")
            mapping = pd.concat([df[col], y], axis=1).groupby(col)[y.name].mean()
            df[col] = df[col].map(mapping).fillna(y.mean())
            info['target_encoded_columns'].append(col)
    # Outlier handling (optional)
    if outlier_method is not None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if verbose:
            print(f"Outlier method: {outlier_method}; cap={cap_outliers}; threshold={outlier_threshold}")
        mask_keep = pd.Series(True, index=df.index)
        for col in num_cols:
            col_vals = df[col]
            if outlier_method == 'iqr':
                Q1 = col_vals.quantile(0.25)
                Q3 = col_vals.quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - outlier_threshold * IQR
                upper = Q3 + outlier_threshold * IQR
                outlier_mask = (col_vals < lower) | (col_vals > upper)
                if cap_outliers:
                    df[col] = col_vals.clip(lower=lower, upper=upper)
                else:
                    mask_keep &= ~outlier_mask
            elif outlier_method in ('zscore','z-score','z_score'):
                mean = col_vals.mean()
                std = col_vals.std(ddof=0)
                if std == 0 or np.isnan(std):
                    continue
                z = (col_vals - mean) / std
                outlier_mask = z.abs() > outlier_threshold
                if cap_outliers:
                    df[col] = col_vals.where(~outlier_mask, np.sign(z) * outlier_threshold * std + mean)
                else:
                    mask_keep &= ~outlier_mask
        if not cap_outliers:
            before = df.shape[0]
            df = df[mask_keep].reset_index(drop=True)
            if y is not None:
                y = y.loc[mask_keep].reset_index(drop=True)
            dropped = before - df.shape[0]
            info['outlier_dropped_count'] = int(dropped)
            if verbose:
                print(f"Dropped {dropped} rows due to outliers")
    # Identify numeric and categorical columns (post-transformations)
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = df.select_dtypes(include=['object','category','string']).columns.tolist()
    # Multi-column LabelEncoder (fits a separate LabelEncoder per categorical column)
    class MultiLabelEncoder(BaseEstimator, TransformerMixin):
        def __init__(self, feature_names=None):
            self.encoders_ = []
            self.feature_names = feature_names
            self.feature_names_in_ = feature_names
        def fit(self, X, y=None):
            if self.feature_names_in_ is None:
                self.feature_names_in_ = [f'cat_{i}' for i in range(X.shape[1])]
            df_cat = pd.DataFrame(X, columns=self.feature_names_in_)
            self.encoders_ = []
            for col in df_cat.columns:
                le = LabelEncoder()
                le.fit(df_cat[col].astype(str))
                self.encoders_.append(le)
            return self
        def transform(self, X):
            df_cat = pd.DataFrame(X, columns=self.feature_names_in_)
            encoded_cols = []
            for col, le in zip(df_cat.columns, self.encoders_):
                encoded_cols.append(le.transform(df_cat[col].astype(str)))
            return np.vstack(encoded_cols).T
        def get_feature_names_out(self, input_features=None):
            return np.array(input_features if input_features is not None else self.feature_names_in_)
        def set_output(self, *, transform=None):
            return self
        def set_params(self, **params):
            for key, value in params.items():
                setattr(self, key, value)
            return self
        def get_params(self, deep=True):
            return {'feature_names': self.feature_names} 
    # Imputer selection
    if impute_strategy_num == 'knn':
        numeric_imputer = KNNImputer()
    elif impute_strategy_num == 'mode':
        # Custom mode imputer for numeric columns
        class ModeImputer(BaseEstimator, TransformerMixin):
            def fit(self, X, y=None):
                self.fill_ = pd.DataFrame(X).mode().iloc[0].values
                return self
            def transform(self, X):
                X = np.array(X)
                mask = pd.isnull(X)
                X[mask] = np.take(self.fill_, np.where(mask)[1])
                return X
        numeric_imputer = ModeImputer()
    else:
        numeric_imputer = SimpleImputer(strategy=impute_strategy_num)
    categorical_imputer = SimpleImputer(strategy=impute_strategy_cat)
    # Scaler selection
    if scaler == 'standard':
        scaler_obj = StandardScaler()
    elif scaler == 'minmax':
        scaler_obj = MinMaxScaler()
    elif scaler == 'robust':
        scaler_obj = RobustScaler()
    else:
        scaler_obj = 'passthrough'
    # Encoder selection (default: no encoding unless explicitly requested)
    if encode_categorical in (None, 'none', 'passthrough'):
        encoder = 'passthrough'
    elif encode_categorical == 'onehot':
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    elif encode_categorical == 'ordinal':
        encoder = OrdinalEncoder()
    elif encode_categorical == 'label':
        encoder = MultiLabelEncoder(feature_names=categorical_features)
    else:
        encoder = 'passthrough'
    # Debug: pretty snapshot for selected features and scaling effect
    if verbose:
        dbg_sep = '-' * 88
        dbg_title = ' PREPROCESSING DEBUG INFO '
        print(f"\n{dbg_sep}")
        print(dbg_title.center(88, '-'))
        print(dbg_sep)
        print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
        print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
        print(" ")
        print(f"Scaler object: {scaler_obj}")
        dtype_summary = pd.DataFrame({
            'column': df.columns,
            'dtype': df.dtypes.astype(str).values,
            'nulls': df.isna().sum().values,
            'unique': [df[c].nunique(dropna=False) for c in df.columns]
        })
        print('\n[Data types / nulls / unique]')
        print(dtype_summary.to_string(index=False))
        if numeric_features:
            print('\n[First 5 rows of numeric columns]')
            print(df[numeric_features].head().to_string(index=False))
            preview_before = pd.DataFrame(
                clone(numeric_imputer).fit_transform(df[numeric_features]),
                columns=numeric_features
            )
            before_stats = preview_before.agg(['min', 'max', 'mean', 'std']).T.add_prefix('before_')
            if scaler_obj != 'passthrough':
                preview_after = pd.DataFrame(
                    clone(scaler_obj).fit_transform(preview_before),
                    columns=numeric_features
                )
                after_stats = preview_after.agg(['min', 'max', 'mean', 'std']).T.add_prefix('after_')
                compare_stats = before_stats.join(after_stats).round(4)
                print('\n[Scaling summary: BEFORE vs AFTER]')
                print(compare_stats.to_string())
            else:
                print('\n[Scaling summary]')
                print('Scaler is passthrough; values unchanged.')
        print(dbg_sep)
    numeric_transformer = Pipeline([
        ('imputer', numeric_imputer),
        ('scaler', scaler_obj),
    ])
    categorical_transformer = Pipeline([
        ('imputer', categorical_imputer),
        ('encoder', encoder)
    ])
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])
    # Fit and transform
    processed_data = preprocessor.fit_transform(df)
    # Feature names
    feature_names = []
    if numeric_features:
        feature_names += numeric_features
    if categorical_features and encode_categorical == 'onehot':
        ohe = preprocessor.named_transformers_['cat'].named_steps['encoder']
        cat_names = ohe.get_feature_names_out(categorical_features)
        feature_names += cat_names.tolist()
    elif categorical_features:
        feature_names += categorical_features
    # DataFrame
    X_processed = pd.DataFrame(processed_data, columns=feature_names)
    # Correlation-based feature removal (optional)
    if remove_correlated:
        if verbose:
            print(f"Removing correlated features with threshold {corr_threshold}")
        numeric_cols_corr = X_processed.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols_corr:
            corr = X_processed[numeric_cols_corr].corr().abs()
            upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            to_drop = [column for column in upper.columns if any(upper[column] > corr_threshold)]
            if to_drop:
                X_processed = X_processed.drop(columns=to_drop)
                info['correlated_dropped'] = to_drop
                feature_names = [f for f in feature_names if f not in to_drop]
                if verbose:
                    print(f"Dropped correlated features ({len(to_drop)}): {to_drop}")
            elif verbose:
                print(f"No correlated numeric features above threshold {corr_threshold}.")
        elif verbose:
            print('Skipped correlation removal: no numeric columns available.')
    # Remove low-variance features if requested
    if remove_low_variance:
        numeric_cols_lv = X_processed.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols_lv:
            selector = VarianceThreshold(threshold=variance_thresh)
            X_num = X_processed[numeric_cols_lv]
            support = selector.fit(X_num).get_support()
            kept_num = [c for i, c in enumerate(numeric_cols_lv) if support[i]]
            dropped = [c for i, c in enumerate(numeric_cols_lv) if not support[i]]
            X_num_kept = pd.DataFrame(selector.transform(X_num), columns=kept_num, index=X_processed.index)
            X_non_num = X_processed.drop(columns=numeric_cols_lv)
            X_processed = pd.concat([X_num_kept, X_non_num], axis=1)
            feature_names = X_processed.columns.tolist()
            info['low_variance_dropped'] = dropped
            if verbose:
                if dropped:
                    print(f"Removed low-variance numeric features ({len(dropped)}) below {variance_thresh}: {dropped}")
                else:
                    print(f"No numeric features below variance threshold {variance_thresh}.")
        elif verbose:
            print('Skipped low-variance removal: no numeric columns available.')
    # Finalize info
    info['final_shape'] = X_processed.shape
    info['feature_names'] = feature_names
    # Always show imputer strategy for numeric imputer
    if isinstance(numeric_imputer, SimpleImputer):
        info['imputer_num'] = f"SimpleImputer(strategy='{numeric_imputer.strategy}')"
    elif numeric_imputer.__class__.__name__ == 'ModeImputer':
        info['imputer_num'] = "ModeImputer(strategy='mode')"
    else:
        info['imputer_num'] = str(numeric_imputer)
    info['imputer_cat'] = str(categorical_imputer)
    info['scaler'] = scaler
    info['encoder'] = encode_categorical

    # ─── Pretty-print a well-separated summary when verbose=True ────────
    if verbose:
        sep = "=" * 72
        print(f"\n{sep}")
        print("  PREPROCESSING INFO  ".center(72, "="))
        print(sep)
        print(f"  Loaded shape         : {info.get('loaded_shape')}")
        print(f"  Final shape          : {info.get('final_shape')}")
        print(f"  Imputer (numeric)    : {info.get('imputer_num')}")
        print(f"  Imputer (categorical): {info.get('imputer_cat')}")
        print(f"  Scaler               : {info.get('scaler')}")
        print(f"  Encoder              : {info.get('encoder')}")
        if info.get('dropped_missing_columns'):
            print(f"  Dropped (high-miss)  : {info['dropped_missing_columns']}")
        if info.get('dropped_user_columns'):
            print(f"  Dropped (user)       : {info['dropped_user_columns']}")
        if info.get('dropped_constant_columns'):
            print(f"  Dropped (constant)   : {info['dropped_constant_columns']}")
        if info.get('correlated_dropped'):
            print(f"  Dropped (correlated) : {info['correlated_dropped']}")
        if info.get('low_variance_dropped'):
            print(f"  Dropped (low-var)    : {info['low_variance_dropped']}")
        if info.get('outlier_method'):
            print(f"  Outlier method       : {info['outlier_method']}  |  dropped rows: {info.get('outlier_dropped_count')}")
        print(sep)

        # Feature Names
        print(f"\n{sep}")
        print("  FEATURE NAMES  ".center(72, "="))
        print(sep)
        print(f"  Total features: {len(feature_names)}")
        for i, name in enumerate(feature_names, 1):
            print(f"  {i:>3}. {name}")
        print(sep)

        # Processed Features (X) — first 5 rows
        print(f"\n{sep}")
        print("  PROCESSED FEATURES (X) — first 5 rows  ".center(72, "="))
        print(sep)
        print(X_processed.head().to_string(index=False))
        print(f"\n  Shape: {X_processed.shape}")
        print(sep)

        # Target Variable (y)
        if y is not None:
            print(f"\n{sep}")
            print("  TARGET VARIABLE (y) — first 10 values  ".center(72, "="))
            print(sep)
            print(y.reset_index(drop=True).head(10).to_string())
            print(f"\n  Shape: {y.shape}")
            print(sep)

    # Return structured output if requested
    if structured_output:
        return {
            'X': X_processed,
            'y': (y.reset_index(drop=True) if y is not None else None),
            'feature_names': feature_names,
            'info': info
        }
    else:
        if y is not None:
            return X_processed, y.reset_index(drop=True), feature_names
        else:
            return X_processed, feature_names

## Train a model using AutoSweep output
Use this cell to preprocess your raw file and then train a baseline ML model on `X` and `y`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score
import pandas as pd

out = AutoSweep(
    file_path=r'D:\sem-6\NN\Housing.csv',
    target_column='price',
    drop_columns=['id', 'timestamp'],
    drop_threshold=0.5,
    impute_strategy_num='mode',
    impute_strategy_cat='most_frequent',        
    encode_categorical='label',
    remove_low_variance=True,
    variance_thresh=0.01,
    verbose=True,
)

X = out['X']
y = out['y']

if y is None:
    raise ValueError('Please pass a valid target_column to AutoSweep for training.')

# 3) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4) Auto-select baseline model type
is_classification = (y.nunique() <= 20) and (str(y.dtype).startswith('int') or str(y.dtype) == 'object' or str(y.dtype) == 'category')

if is_classification:
    model = RandomForestClassifier(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print('Task: Classification')
    print('Accuracy:', round(accuracy_score(y_test, preds), 4))
    print('F1 (weighted):', round(f1_score(y_test, preds, average='weighted'), 4))
else:
    model = RandomForestRegressor(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print('Task: Regression')
    print('MAE:', round(mean_absolute_error(y_test, preds), 4))
    print('R2:', round(r2_score(y_test, preds), 4))

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Total features after preprocessing:', X.shape[1])